In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [13]:
import sqlite3
import pandas as pd

# 1. Load the logistics tables
orders_df = pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv')
items_df = pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv')
customers_df = pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv')

# 2. Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# 3. Write tables to SQLite
orders_df.to_sql('orders', conn, index=False, if_exists='replace')
items_df.to_sql('order_items', conn, index=False, if_exists='replace')
customers_df.to_sql('customers', conn, index=False, if_exists='replace')

# Helper function to run SQL queries and print formatted results
def run_query(query):
    return pd.read_sql_query(query, conn)

print("Tables successfully loaded into SQLite: orders, order_items, customers")

Tables successfully loaded into SQLite: orders, order_items, customers


In [14]:
query_1 = """
WITH DeliveryMetrics AS (
    SELECT 
        c.customer_state,
        o.order_id,
        ROUND(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_delivered_carrier_date), 2) AS transit_days,
        CASE 
            WHEN JULIANDAY(o.order_delivered_customer_date) > JULIANDAY(o.order_estimated_delivery_date) THEN 1
            ELSE 0
        END AS is_delayed
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_delivered_carrier_date IS NOT NULL
)
SELECT 
    customer_state,
    COUNT(order_id) AS total_delivered_orders,
    ROUND(AVG(transit_days), 2) AS avg_transit_days,
    ROUND(100.0 * SUM(is_delayed) / COUNT(order_id), 2) AS delay_rate_pct
FROM DeliveryMetrics
GROUP BY customer_state
HAVING total_delivered_orders >= 500
ORDER BY delay_rate_pct DESC
LIMIT 10;
"""

df_delays = run_query(query_1)
print(df_delays)

  customer_state  total_delivered_orders  avg_transit_days  delay_rate_pct
0             MA                     717             17.97           19.67
1             CE                    1279             17.93           15.32
2             BA                    3256             16.04           14.04
3             RJ                   12350             12.00           13.47
4             PA                     946             20.29           12.37
5             ES                    1995             12.41           12.23
6             MS                     701             12.45           11.55
7             PB                     517             16.92           11.03
8             PE                    1593             15.17           10.80
9             SC                    3546             11.63            9.76


In [15]:
query_2 = """
WITH FulfillmentBreakdown AS (
    SELECT 
        o.order_id,
        c.customer_state,
        ROUND(JULIANDAY(o.order_delivered_carrier_date) - JULIANDAY(o.order_approved_at), 2) AS dispatch_lead_time_days,
        ROUND(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_delivered_carrier_date), 2) AS carrier_transit_days,
        ROUND(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_estimated_delivery_date), 2) AS promise_gap_days,
        CASE 
            WHEN JULIANDAY(o.order_delivered_customer_date) > JULIANDAY(o.order_estimated_delivery_date) THEN 1 
            ELSE 0 
        END AS is_late
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_approved_at IS NOT NULL
      AND o.order_delivered_carrier_date IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
)
SELECT 
    CASE 
        WHEN is_late = 1 THEN 'Delayed Delivery'
        ELSE 'On-Time Delivery'
    END AS delivery_performance,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(dispatch_lead_time_days), 2) AS avg_warehouse_dispatch_days,
    ROUND(AVG(carrier_transit_days), 2) AS avg_carrier_transit_days,
    ROUND(AVG(promise_gap_days), 2) AS avg_promise_variance_days
FROM FulfillmentBreakdown
GROUP BY is_late;
"""

df_breakdown = run_query(query_2)
print(df_breakdown)

  delivery_performance  total_orders  avg_warehouse_dispatch_days  \
0     On-Time Delivery         88630                         2.58   
1     Delayed Delivery          7825                         5.32   

   avg_carrier_transit_days  avg_promise_variance_days  
0                      7.89                     -13.01  
1                     25.68                       9.55  


In [16]:
query_3 = """
WITH OrderFreightTransit AS (
    SELECT 
        o.order_id,
        SUM(i.freight_value) AS total_freight,
        ROUND(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_delivered_carrier_date), 2) AS transit_days
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_delivered_carrier_date IS NOT NULL
    GROUP BY o.order_id
),
FreightTiers AS (
    SELECT 
        order_id,
        total_freight,
        transit_days,
        NTILE(4) OVER (ORDER BY total_freight) AS freight_tier
    FROM OrderFreightTransit
)
SELECT 
    CASE freight_tier
        WHEN 1 THEN 'Tier 1 - Low Cost'
        WHEN 2 THEN 'Tier 2 - Mid-Low Cost'
        WHEN 3 THEN 'Tier 3 - Mid-High Cost'
        WHEN 4 THEN 'Tier 4 - Premium Freight'
    END AS freight_tier_label,
    COUNT(order_id) AS order_volume,
    ROUND(MIN(total_freight), 2) AS min_freight,
    ROUND(MAX(total_freight), 2) AS max_freight,
    ROUND(AVG(transit_days), 2) AS avg_transit_days
FROM FreightTiers
GROUP BY freight_tier
ORDER BY freight_tier;
"""

df_freight = run_query(query_3)
print(df_freight)

         freight_tier_label  order_volume  min_freight  max_freight  \
0         Tier 1 - Low Cost         24118         0.00        13.85   
1     Tier 2 - Mid-Low Cost         24117        13.85        17.17   
2    Tier 3 - Mid-High Cost         24117        17.17        24.02   
3  Tier 4 - Premium Freight         24117        24.02      1794.96   

   avg_transit_days  
0              4.79  
1              9.84  
2             10.94  
3             11.76  
